In [0]:
from pyspark.sql.functions import row_number, floor, col, avg, abs, round, lower
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text(
    "silver_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "silver_schema",
    "artemzharkov10_silver"
)

dbutils.widgets.text(
    "gold_catalog",
    "dbr_dev")
dbutils.widgets.text(
    "gold_schema",
    "artemzharkov10_gold"
)

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")


In [0]:
df_silver = spark.read.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.gdlet_history_silver")

KEY_WORDS = "crypto|bitcoin|btc|elon|musk|trump"

df_filtered = df_silver.filter(lower(col("SourceUrl")).rlike(KEY_WORDS))

df_gold_news = df_filtered.select(
    col("SqlDate").alias("NewsDate"),
    col("SourceUrl").alias("URL"),
    col("Actor1Name"),
    col("Actor2Name"),
    col("GoldsteinScale").alias("NewsTone"),
    col("AvgTone").alias("AverageTone")
)

(df_gold_news.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.bitcoin_news_gold"))
